<a href="https://colab.research.google.com/github/Pujasatish001/DS-Lab-Experiments/blob/main/DS_Lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Experiment 13 : 23-11-2026

import hashlib
import itertools

# Configuration
BLOCK_SIZE = 20
REPLICATION_FACTOR = 3

# Create six DataNodes in two racks
datanodes = {
    "DN1": {"rack": "Rack1", "storage": {}},
    "DN2": {"rack": "Rack1", "storage": {}},
    "DN3": {"rack": "Rack1", "storage": {}},
    "DN4": {"rack": "Rack2", "storage": {}},
    "DN5": {"rack": "Rack2", "storage": {}},
    "DN6": {"rack": "Rack2", "storage": {}}
}

# NameNode metadata
namenode = {
    "file_to_blocks": {},
    "block_to_datanodes": {}
}

# Track active DataNodes
active_nodes = list(datanodes.keys())


# Function to calculate checksum
def checksum(data):
    return hashlib.sha256(data).hexdigest()


# Select the DataNode with minimum stored blocks
def select_node(rack=None, exclude=None):
    exclude = exclude or []

    candidates = [
        node for node in active_nodes
        if node not in exclude
        and (rack is None or datanodes[node]["rack"] == rack)
    ]

    if not candidates:
        raise Exception("Not enough active DataNodes!")

    return min(
        candidates,
        key=lambda node: len(datanodes[node]["storage"])
    )


# Function to store a block with three replicas
def store_block(block_id, block_data):

    # First replica
    first = select_node()
    selected = [first]

    # Second replica on another rack
    first_rack = datanodes[first]["rack"]
    second_rack = "Rack2" if first_rack == "Rack1" else "Rack1"

    second = select_node(second_rack, selected)
    selected.append(second)

    # Third replica on the second replica's rack
    third = select_node(second_rack, selected)
    selected.append(third)

    # Store replicas
    for node in selected:
        datanodes[node]["storage"][block_id] = block_data

    namenode["block_to_datanodes"][block_id] = selected

    return selected


# Function to upload a file into HDFS
def upload_file(filename, data):

    print("\n--- FILE UPLOAD ---")

    # Convert text into bytes
    file_data = data.encode("utf-8")

    # Split file into fixed-size blocks
    blocks = [
        file_data[i:i + BLOCK_SIZE]
        for i in range(0, len(file_data), BLOCK_SIZE)
    ]

    block_ids = []

    for i, block in enumerate(blocks, start=1):

        block_id = "blk_" + str(i)
        block_ids.append(block_id)

        print("\nBlock:", block_id)
        print("Data:", block.decode("utf-8"))

        nodes = store_block(block_id, block)

        print("Replicas stored in:", nodes)

    # Update NameNode metadata
    namenode["file_to_blocks"][filename] = block_ids

    print("\nFile uploaded successfully!")


# Function to display block placement
def show_metadata():

    print("\n--- NAMENODE METADATA ---")

    for filename, blocks in namenode["file_to_blocks"].items():
        print("\nFile:", filename)

        for block in blocks:
            print(
                block,
                "->",
                namenode["block_to_datanodes"][block]
            )


# Function to read a file from HDFS
def read_file(filename):

    print("\n--- READING FILE ---")

    result = b""

    for block_id in namenode["file_to_blocks"][filename]:

        # Find an available replica
        nodes = namenode["block_to_datanodes"][block_id]

        for node in nodes:
            if node in active_nodes:
                result += datanodes[node]["storage"][block_id]
                break
        else:
            raise Exception("Block unavailable: " + block_id)

    print("File content:", result.decode("utf-8"))

    return result


# Function to simulate DataNode failure
def fail_datanode(node):

    print("\n--- DATANODE FAILURE ---")

    if node not in active_nodes:
        print(node, "is already inactive!")
        return

    active_nodes.remove(node)

    print(node, "has failed!")

    # Remove the failed node from NameNode locations
    for block_id in namenode["block_to_datanodes"]:

        locations = namenode["block_to_datanodes"][block_id]

        if node in locations:
            locations.remove(node)

    # Re-replicate blocks that have fewer than 3 copies
    for block_id in namenode["block_to_datanodes"]:

        locations = namenode["block_to_datanodes"][block_id]

        while len(locations) < REPLICATION_FACTOR:

            # Choose rack with fewer existing replicas
            rack_counts = {
                "Rack1": 0,
                "Rack2": 0
            }

            for existing in locations:
                rack = datanodes[existing]["rack"]
                rack_counts[rack] += 1

            target_rack = min(
                rack_counts,
                key=rack_counts.get
            )

            new_node = select_node(
                target_rack,
                locations
            )

            # Copy data from an existing healthy replica
            source_node = locations[0]

            block_data = datanodes[source_node]["storage"][block_id]

            datanodes[new_node]["storage"][block_id] = block_data

            locations.append(new_node)

            print(
                block_id,
                "re-replicated to",
                new_node
            )


# Main program
if __name__ == "__main__":

    filename = "sensor_logs.txt"

    data = (
        "Smart city sensor data: "
        "Temperature=32C, Humidity=65%, "
        "Traffic=Normal, AirQuality=Good, "
        "WaterLevel=Safe, Electricity=Stable."
    )

    # Calculate original checksum
    original_checksum = checksum(data.encode("utf-8"))

    print("Original Checksum:", original_checksum)

    # Upload file
    upload_file(filename, data)

    # Display NameNode metadata
    show_metadata()

    # Read file
    downloaded_data = read_file(filename)

    # Verify checksum
    downloaded_checksum = checksum(downloaded_data)

    print("\n--- CHECKSUM VERIFICATION ---")

    if original_checksum == downloaded_checksum:
        print("Checksum Matched! File is healthy.")
    else:
        print("Checksum Mismatch! File is corrupted.")

    # Simulate failure of DN1
    fail_datanode("DN1")

    # Read file after failure
    recovered_data = read_file(filename)

    # Verify file after recovery
    print("\n--- FINAL HEALTH CHECK ---")

    if checksum(recovered_data) == original_checksum:
        print("File recovered successfully!")
        print("All data is available after DataNode failure.")
    else:
        print("File recovery failed!")

    # Display final metadata
    show_metadata()

Original Checksum: 561b7f285e5552c24021d10c2fff3f118d0c13ac6a2e3b400d46a71aa1bd3ca1

--- FILE UPLOAD ---

Block: blk_1
Data: Smart city sensor da
Replicas stored in: ['DN1', 'DN4', 'DN5']

Block: blk_2
Data: ta: Temperature=32C,
Replicas stored in: ['DN2', 'DN6', 'DN4']

Block: blk_3
Data:  Humidity=65%, Traff
Replicas stored in: ['DN3', 'DN5', 'DN6']

Block: blk_4
Data: ic=Normal, AirQualit
Replicas stored in: ['DN1', 'DN4', 'DN5']

Block: blk_5
Data: y=Good, WaterLevel=S
Replicas stored in: ['DN2', 'DN6', 'DN4']

Block: blk_6
Data: afe, Electricity=Sta
Replicas stored in: ['DN3', 'DN5', 'DN6']

Block: blk_7
Data: ble.
Replicas stored in: ['DN1', 'DN4', 'DN5']

File uploaded successfully!

--- NAMENODE METADATA ---

File: sensor_logs.txt
blk_1 -> ['DN1', 'DN4', 'DN5']
blk_2 -> ['DN2', 'DN6', 'DN4']
blk_3 -> ['DN3', 'DN5', 'DN6']
blk_4 -> ['DN1', 'DN4', 'DN5']
blk_5 -> ['DN2', 'DN6', 'DN4']
blk_6 -> ['DN3', 'DN5', 'DN6']
blk_7 -> ['DN1', 'DN4', 'DN5']

--- READING FILE ---
File content

In [ ]:
#Experiment 14: 23-11-2026

import re
from collections import defaultdict

# Input headlines divided into 3 splits (nodes)
splits = [
    [
        "AI is changing the world",
        "Data science is the future of AI"
    ],
    [
        "AI helps in data analysis",
        "The world uses data science"
    ],
    [
        "Machine learning is part of AI",
        "Data science uses machine learning"
    ]
]


# MAPPER FUNCTION
def mapper(lines):
    result = []

    for line in lines:
        words = re.findall(r'\b[a-zA-Z]+\b', line.lower())

        for word in words:
            result.append((word, 1))

    return result


# COMBINER FUNCTION
def combiner(mapped_data):
    combined = defaultdict(int)

    for word, count in mapped_data:
        combined[word] += count

    return list(combined.items())


# REDUCER FUNCTION
def reducer(shuffled_data):
    result = {}

    for word, counts in shuffled_data.items():
        result[word] = sum(counts)

    return result


# MAIN PROGRAM
if __name__ == "__main__":

    print("========== MAP PHASE ==========")

    mapped_results = []

    for i, split in enumerate(splits, start=1):
        mapped = mapper(split)
        mapped_results.append(mapped)

        print("\nSplit", i)
        print(mapped)

    # COMBINER PHASE
    print("\n========== COMBINER PHASE ==========")

    combined_results = []

    for i, mapped in enumerate(mapped_results, start=1):
        combined = combiner(mapped)
        combined_results.append(combined)

        print("\nCombiner", i)
        print(combined)

    # SHUFFLE AND SORT PHASE
    print("\n========== SHUFFLE AND SORT PHASE ==========")

    shuffled = defaultdict(list)

    for combined in combined_results:
        for word, count in combined:
            shuffled[word].append(count)

    # Sort words alphabetically
    shuffled = dict(sorted(shuffled.items()))

    for word, counts in shuffled.items():
        print(word, "->", counts)

    # REDUCE PHASE
    print("\n========== REDUCE PHASE ==========")

    final_result = reducer(shuffled)

    for word, count in final_result.items():
        print(word, "->", count)

    # FINAL STATISTICS
    print("\n========== FINAL STATISTICS ==========")

    print("Total distinct words:", len(final_result))

    print("Top 5 frequent words:")

    top_words = sorted(
        final_result.items(),
        key=lambda x: (-x[1], x[0])
    )[:5]

    for word, count in top_words:
        print(word, "->", count)

========== MAP PHASE ==========

Split 1
[('ai', 1), ('is', 1), ('changing', 1), ('the', 1), ('world', 1), ('data', 1), ('science', 1), ('is', 1), ('the', 1), ('future', 1), ('of', 1), ('ai', 1)]

Split 2
[('ai', 1), ('helps', 1), ('in', 1), ('data', 1), ('analysis', 1), ('the', 1), ('world', 1), ('uses', 1), ('data', 1), ('science', 1)]

Split 3
[('machine', 1), ('learning', 1), ('is', 1), ('part', 1), ('of', 1), ('ai', 1), ('data', 1), ('science', 1), ('uses', 1), ('machine', 1), ('learning', 1)]

========== COMBINER PHASE ==========

Combiner 1
[('ai', 2), ('is', 2), ('changing', 1), ('the', 2), ('world', 1), ('data', 1), ('science', 1), ('future', 1), ('of', 1)]

Combiner 2
[('ai', 1), ('helps', 1), ('in', 1), ('data', 2), ('analysis', 1), ('the', 1), ('world', 1), ('uses', 1), ('science', 1)]

Combiner 3
[('machine', 2), ('learning', 2), ('is', 1), ('part', 1), ('of', 1), ('ai', 1), ('data', 1), ('science', 1), ('uses', 1)]

========== SHUFFLE AND SORT PHASE ==========
ai -> [2, 1

In [ ]:
#Experiment 15: 24-11-2026

import numpy as np
from collections import defaultdict

# Step 1: Define the link matrix of 5 web pages
# Column j represents links FROM page j
# Row i represents links TO page i

links = {
    0: [1, 2, 3],
    1: [0, 4],
    2: [3],
    3: [1, 2],
    4: [0, 2, 3]
}

N = 5

# Step 2: Create sparse matrix records
records = []

for j in range(N):
    out_degree = len(links[j])

    for i in links[j]:
        m_ij = 1 / out_degree
        records.append((i, j, m_ij))

# Store matrix records in a separate file
with open("matrix.txt", "w") as f:
    for record in records:
        f.write(f"{record[0]} {record[1]} {record[2]}\n")

# Store initial rank vector in a separate file
v = np.ones(N) / N

with open("vector.txt", "w") as f:
    for value in v:
        f.write(str(value) + "\n")

print("Sparse Matrix Records:")
for record in records:
    print(record)


# Step 3: MAPPER FUNCTION
def mapper(records, vector):

    mapped = []

    for i, j, m_ij in records:
        value = m_ij * vector[j]
        mapped.append((i, value))

    return mapped


# Step 4: REDUCER FUNCTION
def reducer(shuffled):

    result = {}

    for i, values in shuffled.items():
        result[i] = sum(values)

    return result


# Step 5: MapReduce Job
def mapreduce(records, vector):

    print("\n--- MAP PHASE ---")

    mapped = mapper(records, vector)

    for item in mapped:
        print(item)

    print("\n--- SHUFFLE AND SORT PHASE ---")

    shuffled = defaultdict(list)

    for i, value in mapped:
        shuffled[i].append(value)

    shuffled = dict(sorted(shuffled.items()))

    for key, values in shuffled.items():
        print(key, "->", values)

    print("\n--- REDUCE PHASE ---")

    reduced = reducer(shuffled)

    result = np.zeros(N)

    for i, value in reduced.items():
        result[i] = value
        print("P" + str(i), "->", value)

    return result


# Step 6: Create the matrix for NumPy verification
M = np.zeros((N, N))

for i, j, value in records:
    M[i, j] = value


# Step 7: Perform one MapReduce iteration
print("\n========== FIRST ITERATION ==========")

x = mapreduce(records, v)

# Verify using NumPy
numpy_result = M @ v

print("\nMapReduce Result:", x)
print("NumPy Result:", numpy_result)

print("Verification:", np.allclose(x, numpy_result))


# Step 8: Repeat PageRank until convergence
print("\n========== PAGERANK CONVERGENCE ==========")

v = np.ones(N) / N

tolerance = 1e-10
max_iterations = 1000

for iteration in range(1, max_iterations + 1):

    new_v = mapreduce(records, v)

    print("\nIteration", iteration)
    print("PageRank:", new_v)

    if np.linalg.norm(new_v - v, 1) < tolerance:
        v = new_v
        print("\nPageRank Converged!")
        break

    v = new_v

# Final result
print("\n========== FINAL PAGE RANK ==========")

for i, rank in enumerate(v):
    print("P" + str(i), ":", round(rank, 6))

print("\nTotal Rank:", sum(v))
print("Total Iterations:", iteration)

Sparse Matrix Records:
(1, 0, 0.3333333333333333)
(2, 0, 0.3333333333333333)
(3, 0, 0.3333333333333333)
(0, 1, 0.5)
(4, 1, 0.5)
(3, 2, 1.0)
(1, 3, 0.5)
(2, 3, 0.5)
(0, 4, 0.3333333333333333)
(2, 4, 0.3333333333333333)
(3, 4, 0.3333333333333333)

========== FIRST ITERATION ==========

--- MAP PHASE ---
(1, np.float64(0.06666666666666667))
(2, np.float64(0.06666666666666667))
(3, np.float64(0.06666666666666667))
(0, np.float64(0.1))
(4, np.float64(0.1))
(3, np.float64(0.2))
(1, np.float64(0.1))
(2, np.float64(0.1))
(0, np.float64(0.06666666666666667))
(2, np.float64(0.06666666666666667))
(3, np.float64(0.06666666666666667))

--- SHUFFLE AND SORT PHASE ---
0 -> [np.float64(0.1), np.float64(0.06666666666666667)]
1 -> [np.float64(0.06666666666666667), np.float64(0.1)]
2 -> [np.float64(0.06666666666666667), np.float64(0.1), np.float64(0.06666666666666667)]
3 -> [np.float64(0.06666666666666667), np.float64(0.2), np.float64(0.06666666666666667)]
4 -> [np.float64(0.1)]

--- REDUCE PHASE ---
P0 

In [ ]:
#Experiment 16: 24-11-2026

from collections import defaultdict

# CUSTOMER DATA
customers = [
    (1, "Arun", "Chennai"),
    (2, "Priya", "Bangalore"),
    (3, "Rahul", "Mumbai"),
    (4, "Divya", "Chennai"),
    (5, "Karthik", "Delhi")
]

# ORDER DATA
orders = [
    (101, 1, 5000),
    (102, 2, 3000),
    (103, 1, 2000),
    (104, 3, 7000),
    (105, 4, 4000),
    (106, 2, 6000),
    (107, 5, 8000),
    (108, 8, 2500)
]


# =====================================
# JOB 1: REDUCE-SIDE JOIN
# =====================================

# MAPPER FUNCTION
def job1_mapper(customers, orders):

    mapped = []

    # Emit customer records
    for cust_id, name, city in customers:
        mapped.append(
            (cust_id, ("CUSTOMER", name, city))
        )

    # Emit order records
    for order_id, cust_id, amount in orders:
        mapped.append(
            (cust_id, ("ORDER", order_id, amount))
        )

    return mapped


# REDUCER FUNCTION
def job1_reducer(shuffled):

    joined_records = []

    for cust_id, records in shuffled.items():

        customer = None
        customer_orders = []

        # Separate customer and order records
        for record in records:

            if record[0] == "CUSTOMER":
                customer = record

            elif record[0] == "ORDER":
                customer_orders.append(record)

        # Perform INNER JOIN
        if customer is not None:

            name = customer[1]
            city = customer[2]

            for order in customer_orders:

                order_id = order[1]
                amount = order[2]

                joined_records.append(
                    (order_id, cust_id, name, city, amount)
                )

    return joined_records


# =====================================
# JOB 2: GROUPING AND AGGREGATION
# =====================================

# MAPPER FUNCTION
def job2_mapper(joined_records):

    mapped = []

    for order_id, cust_id, name, city, amount in joined_records:

        mapped.append((city, amount))

    return mapped


# REDUCER FUNCTION
def job2_reducer(shuffled):

    result = {}

    for city, amounts in shuffled.items():

        total = sum(amounts)
        count = len(amounts)
        maximum = max(amounts)
        average = total / count

        result[city] = {
            "Total Revenue": total,
            "Number of Orders": count,
            "Maximum Order": maximum,
            "Average Order": average
        }

    return result


# =====================================
# MAIN PROGRAM
# =====================================

if __name__ == "__main__":

    # JOB 1 - MAP PHASE
    print("\n========== JOB 1: MAP PHASE ==========")

    mapped1 = job1_mapper(customers, orders)

    for item in mapped1:
        print(item)

    # JOB 1 - SHUFFLE AND SORT
    print("\n========== JOB 1: SHUFFLE AND SORT ==========")

    shuffled1 = defaultdict(list)

    for key, value in mapped1:
        shuffled1[key].append(value)

    shuffled1 = dict(sorted(shuffled1.items()))

    for key, values in shuffled1.items():
        print(key, "->", values)

    # JOB 1 - REDUCE PHASE
    print("\n========== JOB 1: REDUCE / JOIN ==========")

    joined = job1_reducer(shuffled1)

    for record in joined:
        print(record)

    # JOB 2 - MAP PHASE
    print("\n========== JOB 2: MAP PHASE ==========")

    mapped2 = job2_mapper(joined)

    for item in mapped2:
        print(item)

    # JOB 2 - SHUFFLE AND SORT
    print("\n========== JOB 2: SHUFFLE AND SORT ==========")

    shuffled2 = defaultdict(list)

    for city, amount in mapped2:
        shuffled2[city].append(amount)

    shuffled2 = dict(sorted(shuffled2.items()))

    for city, amounts in shuffled2.items():
        print(city, "->", amounts)

    # JOB 2 - REDUCE PHASE
    print("\n========== JOB 2: REDUCE / AGGREGATION ==========")

    final_result = job2_reducer(shuffled2)

    for city, values in final_result.items():

        print("\nCity:", city)

        for key, value in values.items():
            print(key, ":", value)

    # FINAL REPORT
    print("\n========== FINAL CITY-WISE SALES REPORT ==========")

    print(
        "{:<15} {:<15} {:<12} {:<15} {:<15}".format(
            "City", "Revenue", "Orders", "Maximum", "Average"
        )
    )

    for city, values in final_result.items():

        print(
            "{:<15} {:<15} {:<12} {:<15} {:<15.2f}".format(
                city,
                values["Total Revenue"],
                values["Number of Orders"],
                values["Maximum Order"],
                values["Average Order"]
            )
        )


========== JOB 1: MAP PHASE ==========
(1, ('CUSTOMER', 'Arun', 'Chennai'))
(2, ('CUSTOMER', 'Priya', 'Bangalore'))
(3, ('CUSTOMER', 'Rahul', 'Mumbai'))
(4, ('CUSTOMER', 'Divya', 'Chennai'))
(5, ('CUSTOMER', 'Karthik', 'Delhi'))
(1, ('ORDER', 101, 5000))
(2, ('ORDER', 102, 3000))
(1, ('ORDER', 103, 2000))
(3, ('ORDER', 104, 7000))
(4, ('ORDER', 105, 4000))
(2, ('ORDER', 106, 6000))
(5, ('ORDER', 107, 8000))
(8, ('ORDER', 108, 2500))

========== JOB 1: SHUFFLE AND SORT ==========
1 -> [('CUSTOMER', 'Arun', 'Chennai'), ('ORDER', 101, 5000), ('ORDER', 103, 2000)]
2 -> [('CUSTOMER', 'Priya', 'Bangalore'), ('ORDER', 102, 3000), ('ORDER', 106, 6000)]
3 -> [('CUSTOMER', 'Rahul', 'Mumbai'), ('ORDER', 104, 7000)]
4 -> [('CUSTOMER', 'Divya', 'Chennai'), ('ORDER', 105, 4000)]
5 -> [('CUSTOMER', 'Karthik', 'Delhi'), ('ORDER', 107, 8000)]
8 -> [('ORDER', 108, 2500)]

========== JOB 1: REDUCE / JOIN ==========
(101, 1, 'Arun', 'Chennai', 5000)
(103, 1, 'Arun', 'Chennai', 2000)
(102, 2, 'Priya', 'Ba